# Gamerdust — Guía técnica y funcional

Gamerdust es una **aplicación web que ayuda a diseñar la narrativa de un videojuego**: guarda la historia, los personajes, los lugares y las reglas del mundo, deja escribir el guion escena por escena, dibuja los diálogos ramificados y, con ayuda de una **IA emocional**, avisa cuando una escena o un personaje no encaja con lo que el autor definió.

Esta guía está pensada para alguien que **llega por primera vez** al proyecto. No hace falta saber de IA ni de Laravel: cada concepto se explica con palabras sencillas.

**Fecha de revisión:** 2026-11-16 · **Tests automáticos:** 112 (todos pasan).

**Cómo leerla**

| Si quieres… | Lee |
|---|---|
| Entender qué hace la app | Secciones 1 a 3 |
| Ver cómo está construida | Secciones 4 a 7 |
| Entender la IA emocional (la parte estrella) | **Sección 8** |
| Instalarla y probarla | Sección 10 |
| Saber qué falta | Sección 11 |

---

**Qué cambió respecto a la versión anterior (2026-10-06)**

- Ahora hay un **estudio de proyecto** con menú lateral: Panel, Biblia Narrativa, Guion, Escenas, Diálogos, Mecánicas, Colaboración, Análisis IA y Reportes.
- **IA emocional real** con Groq (antes todo era simulado por palabras clave): análisis de escenas, gráficos de evolución, borrador de personajes y refinado de diálogos.
- **Interfaz bilingüe** (español e inglés).
- El editor de ramificación (estilo Twine) ya está integrado en el estudio.

## 1. La app en un minuto

Imagina que escribes un videojuego y necesitas llevar el control de todo. Gamerdust te da un espacio de trabajo por proyecto con estas herramientas:

| Herramienta | Para qué sirve (en una frase) |
|---|---|
| **Biblia Narrativa** | El "libro de consulta" de tu historia: esencia, personajes, lugares y reglas del mundo. |
| **Guion** | Escribes cada escena en bloques (acción, personaje, diálogo…) como en un guion de cine. |
| **Escenas** | Lista ordenada de escenas con su conflicto, emoción esperada y coste estimado. |
| **Diálogos** | Un lienzo con cajas y flechas para diseñar conversaciones donde el jugador elige. |
| **Mecánicas** | Qué puede hacer el jugador (decidir, explorar, gestionar recursos) y cómo afecta a la historia. |
| **Colaboración** | Comentarios del equipo, opcionalmente ligados a una escena. |
| **Análisis IA** | Métricas, avisos de coherencia y gráficos de cómo evolucionan las emociones. |
| **Reportes** | Resumen del proyecto y descarga en Markdown. |

**Idea clave:** la IA **propone, nunca decide**. No cambia tu guion ni guarda nada sin que tú lo confirmes.

## 2. Recorrido de una persona usuaria

1. Entra a `/` (portada) y se **registra** (`/register`) o **inicia sesión** (`/login`).
2. En `/projects` ve sus proyectos: busca por título, filtra favoritos y cambia entre cuadrícula y lista.
3. **Crea un proyecto** con nombre, género, idioma de la narrativa (español o inglés) y descripción.
4. Entra al **estudio del proyecto** (`/projects/{id}`). El menú lateral lleva a cada sección.
5. En la **Biblia Narrativa** define la esencia (premisa, conflicto, tono) y crea personajes, lugares y reglas.
6. En el **Guion** escribe escenas, vincula personajes y lugar, y pulsa **Analizar emociones**.
7. Revisa en **Análisis IA** los avisos y los gráficos de evolución emocional.
8. Descarga el resumen en **Reportes**.

Para probar rápido existe un usuario de ejemplo (se crea con `php artisan migrate --seed`): `test@example.com` / `password`, con el proyecto *Dustfall Protocol*.

## 3. Las secciones del estudio, una por una

El menú lateral se define en `PROJECT_SECTIONS` (`resources/js/layouts/components/ProjectSidebar.tsx`). En escritorio se puede plegar; en móvil se abre como panel lateral.

### Panel (`/projects/{id}`)
Resumen del proyecto: estadísticas, restricciones creativas (tono, mundo, rol del jugador…), avisos de coherencia y estimaciones de coste. **No llama a la IA**: aquí solo se usan reglas simples.

### Biblia Narrativa (`/bible`)
Cuatro pestañas:

| Pestaña | Qué guarda |
|---|---|
| **Esencia** | Premisa, conflicto principal, tema, tono y ambientación. |
| **Personajes** | Personalidad, motivación, miedos, deseos, **límites** (lo que nunca haría), concept art y su **estado emocional inicial**. |
| **Lugares** | Tipo de lugar, acceso, jerarquía (un lugar dentro de otro) y vínculos con reglas. |
| **Reglas del mundo** | Enunciado, **rigidez** (inquebrantable, rompible a un coste, flexible), origen, consecuencia y excepciones. |

Todo se enlaza: un lugar tiene reglas, un personaje puede *romper* o *imponer* una regla, una escena ocurre en un lugar. **Estos vínculos son los que la IA usa como contexto.**

### Guion (`/script`)
Editor de escenas por bloques. Detalle en la sección 6.

### Escenas (`/scenes`)
Lista clásica de escenas: título, orden, resumen, conflicto, emoción esperada, interés (1–10) y decisión del jugador. Al guardar se calcula un **coste narrativo** (bajo, medio, alto).

### Diálogos (`/branching`)
Lienzo de diálogos ramificados. Detalle en la sección 7.

### Mecánicas (`/mechanics`)
Documento de diseño de cada mecánica: tipo (decisión, exploración, recurso), icono, acciones jugables, restricciones, conflicto y las escenas y personajes implicados.

### Colaboración (`/collaboration`)
Comentarios del proyecto. Cada uno puede apuntar a una escena concreta.

### Análisis IA (`/analysis`)
Dos pestañas: **métricas y coherencia** y **evolución emocional** (gráficos). Detalle en la sección 8.

### Reportes (`/reports`)
Conteo de escenas, personajes, lugares y mecánicas, y descarga de todo el proyecto en un archivo **Markdown**.

## 4. Cómo está construida (vista general)

```
app/ resources/ ...    ← LA app: Laravel (PHP) + React (TypeScript), en la raíz del repositorio
docs/                  ← documentación (este cuaderno, modelo-asistencia-emocional.md, despliegue.md)
compose.yaml           ← solo MySQL para desarrollo
```

La raíz del repositorio **es** la app Laravel. Las rutas de este cuaderno son relativas a esa raíz.

### Las piezas y cómo se hablan

```
 Navegador (React)  ◄──── Inertia ────►  Laravel (PHP)  ◄────►  MySQL
   páginas y formularios   "pegamento"    rutas, reglas,         datos
                                          validación, IA
                                               │
                                               ▼
                                         Groq (IA externa)
```

| Capa | Tecnología | Dónde |
|---|---|---|
| Servidor | Laravel 13, PHP 8.4+ | `app/`, `routes/web.php` |
| Interfaz | React 19 + TypeScript | `resources/js/` |
| Pegamento | **Inertia**: Laravel entrega páginas React sin crear una API aparte | `resources/js/app/resolve-page.ts` |
| Estilos | Tailwind CSS 4 | `resources/css/` |
| Compilación | Vite 8 | `vite.config.js` |
| Base de datos | MySQL (en los tests, SQLite en memoria) | `database/migrations/` |
| Gráficos | Recharts (cargado solo cuando se necesita) | `features/analysis/components/EvolutionCharts.tsx` |
| Grafo de diálogos | `@xyflow/react` | `features/branching/` |

### Cómo se conecta una página

El controlador devuelve `Inertia::render('Projects/Index')` y el frontend busca el archivo `resources/js/features/projects/pages/Index.tsx`. La regla: **nombre de la página → `features/<módulo en minúsculas>/pages/<Página>.tsx`**. Si una página nueva no sigue esa ruta, falla al abrirse.

### Organización del frontend

```
resources/js/
├── app.tsx              arranque de Inertia y React
├── layouts/             marco de las páginas (cabecera, menú lateral del estudio)
├── features/            una carpeta por módulo: pages/, components/, <módulo>.types.ts
│   ├── projects  bible  characters  lore  scenes  script
│   ├── branching  dialogues  mechanics  collaboration
│   └── analysis  reports  landing  auth  account
└── shared/lib/          utilidades: i18n.ts, api.ts, styles.ts
```

### Dos formas de comunicarse con el servidor

- **Páginas normales** (proyectos, biblia, mecánicas…): formularios Inertia. Laravel valida, guarda y vuelve a pintar la página.
- **Editores dinámicos** (Guion, Diálogos, análisis de IA): peticiones **JSON** con `fetch` (`shared/lib/api.ts`), para guardar sin recargar la página.

## 5. Datos, idiomas y seguridad

### 5.1 Las tablas de la base de datos (mapa)

Todo cuelga de un **proyecto**, que pertenece a un **usuario**. Si se borra un proyecto, se borra todo lo suyo.

| Grupo | Tablas | Qué guardan |
|---|---|---|
| Base | `users`, `projects`, `project_constraints` | Cuentas, proyectos (género, idioma, favorito…) y sus restricciones creativas (1 fila por proyecto) |
| Biblia | `characters`, `lore_entries` | Personajes; y lore (lugares, reglas, facciones, objetos…) en una misma tabla distinguida por `type` |
| Guion | `scenes`, `scene_versions`, `character_scene` | Escenas con sus bloques de guion (JSON), historial de versiones y qué personajes salen en cada escena |
| Diálogos | `dialogue_nodes`, `dialogue_options` | Cajas de diálogo y las opciones/flechas entre ellas |
| Mecánicas | `mechanics`, `mechanic_scene`, `mechanic_character` | Mecánicas y sus vínculos |
| Vínculos de la Biblia | `lore_entry_links`, `character_lore_entry`, `lore_entry_scene`, `lore_entry_mechanic` | Quién se relaciona con qué (lugar↔regla, personaje↔regla…) |
| Equipo | `comments` | Comentarios |
| Coherencia y coste | `narrative_validations`, `cost_estimations` | Avisos y estimaciones |
| **IA emocional** | `emotional_analyses`, `character_emotional_states`, `ai_generations`, `ai_feedback` | Resultado de cada análisis, estado emocional de cada personaje por escena, registro de cada llamada a la IA y votos 👍/👎 |

Detalles que conviene saber:

- El **código visible de una escena (E01, E02…) no se guarda**: se calcula según su orden.
- El guion de una escena es un JSON `[{id, kind, text}]`. Los tipos de bloque son `heading`, `action`, `character`, `dialogue` y `parenthetical`.
- Los **lugares y las reglas** son entradas de lore (`type` = `location` o `rule`) con campos propios.
- `projects.language` es el idioma **de la historia** (`es` / `en`), distinto del idioma **de la interfaz**.

### 5.2 Interfaz bilingüe (español e inglés)

- **Nunca se escribe texto visible a mano.** Se usa `__('English text')` y la traducción vive en `lang/es.json`.
- La **clave es el texto en inglés**. No hay `lang/en.json`: en inglés se muestra la propia clave.
- Funciona igual en PHP y en React (mismo catálogo de traducciones).
- El idioma activo es español por defecto; el selector de idioma guarda una cookie `locale`.

### 5.3 Seguridad

- Hay que iniciar sesión para ver cualquier proyecto (`auth`).
- `ProjectPolicy`: solo el **dueño** del proyecto puede verlo o modificarlo.
- Cada recurso hijo (personaje, escena, lugar…) se comprueba contra el proyecto de la URL para que nadie pueda tocar datos de otro proyecto (`abort_unless(..., 404)`).
- Las contraseñas se guardan cifradas (hash) y la sesión se regenera al entrar.
- La **clave de Groq nunca sale del servidor**: el navegador no la ve.
- Las llamadas a la IA tienen un límite propio de **10 por minuto y usuario**.

## 6. El editor de guion

Ruta: `/projects/{id}/script`. Es la herramienta donde más se escribe.

**Qué permite**

- Escribir cada escena en **bloques**: encabezado, acción, personaje, diálogo y acotación.
- Organizar las escenas por **actos**, y vincular a cada una su **lugar** y sus **personajes**.
- **Guardar** con el botón o con `Ctrl/⌘ + S`. Mientras no guardas, trabajas en un **borrador local** con deshacer y rehacer; si intentas salir con cambios pendientes, el navegador avisa.
- **Historial de versiones**: cada guardado crea una versión (se conservan las últimas 30 por escena) y se puede restaurar cualquiera.
- **Imprimir o guardar como PDF** con formato de guion (`/script/print`).
- Comentar la escena desde el propio editor.
- Usar la IA: **Analizar emociones** y **Refinar con IA** (sección 8).

**Protección contra pisarse:** al guardar, el navegador envía la fecha de la última versión que conoce. Si otra persona (u otra pestaña) guardó antes, el servidor responde **409** y no sobrescribe nada.

## 7. El editor de diálogos ramificados

Ruta: `/projects/{id}/branching`. Es un lienzo estilo **Twine**: cada caja es un *pasaje* de diálogo y cada flecha, una opción que el jugador puede elegir.

| Elemento | En la base de datos | Qué es |
|---|---|---|
| Nodo / pasaje | `dialogue_nodes` | Una caja con título, texto, tipo de diálogo, personaje, escena y tono |
| Opción / flecha | `dialogue_options` | Une un nodo con otro; puede llevar texto, consecuencia y condición |
| Posición | `pos_x`, `pos_y` | Dónde está la caja en el lienzo (se conserva) |

**Truco: escribir `{{referencias}}`.** Si en el texto de un nodo escribes:

```text
La puerta se abre. Puedes entrar al {{Archivo subterráneo}} o volver al {{Patio}}.
```

el sistema **crea solo** los nodos "Archivo subterráneo" y "Patio" (si no existían) y las flechas hacia ellos. Si borras la referencia del texto, se borra esa flecha (pero no el nodo destino). Si renombras un nodo, las referencias en los demás textos se actualizan solas.

**Cómo funciona por dentro**

- **El servidor es la fuente de verdad:** cada cambio devuelve el grafo completo y la pantalla lo reemplaza.
- El guardado es **automático** (texto tras ~0,8 s sin escribir; posiciones tras ~0,5 s).
- El editor avisa de problemas: opciones sin destino, nodos aislados, sin entrada o sin salida.

**Alcance:** el editor sirve para **construir y revisar** la estructura. No ejecuta condiciones ni reproduce el diálogo como lo haría el juego. El análisis emocional de estos nodos todavía usa palabras clave (no Groq).

## 8. La IA emocional, explicada para principiantes

> Esta sección resume `docs/modelo-asistencia-emocional.md` en lenguaje sencillo. Allí están los detalles técnicos completos (prompt, esquema, límites y referencias).

### 8.1 ¿Qué problema resuelve?

Un autor puede olvidar que su personaje es miedoso y hacerlo actuar con valentía sin explicación, o saltar de una escena alegre a una desesperada sin transición. La IA **lee la escena**, la compara con lo que el autor escribió en la Biblia y **avisa de lo que no encaja**.

**Qué hace:** estima la emoción de la escena y de cada personaje, detecta cambios bruscos, señala contradicciones con el lugar o las reglas del mundo, y propone sugerencias.

**Qué NO hace:** no reescribe el guion, no guarda nada por su cuenta y no entrena un modelo propio. Usa un modelo de lenguaje general (*gpt-oss* servido por **Groq**) al que se le da buen contexto y se le exige una respuesta ordenada.

### 8.2 La regla de oro

> La IA **solo se llama cuando la persona pulsa un botón**: *Analizar emociones*, *Reanalizar*, *Refinar con IA* o *Borrador con IA*.

Nunca se llama al abrir el Panel, al guardar ni en segundo plano. Así el coste está controlado y el autor siempre decide.

### 8.3 El camino de un análisis, paso a paso

```
 Pulsas «Analizar emociones»
          │
          ▼
 ¿Hay cambios sin guardar?  ── sí ──► Aviso: guarda primero (no se llama a la IA)
          │ no
          ▼
 ¿Ya analicé exactamente este contenido?  ── sí ──► Muestra el resultado guardado (gratis)
          │ no
          ▼
 Se arma el CONTEXTO desde la Biblia
 (esencia, lugar, reglas, personajes, estado previo, escena anterior)
          │
          ▼
 Groq (gpt-oss-120b) responde en JSON con formato estricto
          │  ── si falla ──► reintento con gpt-oss-20b ── si vuelve a fallar ──► palabras clave
          ▼
 Validación en PHP (se comprueba y se corrige lo que venga raro)
          │
          ▼
 Traductor emocional  (números → palabras que entiende un autor)
          │
          ▼
 Regla de cambio brusco  (la regla decide, la IA explica)
          │
          ▼
 Se guarda y se muestra en el panel del Guion y en los gráficos
```

### 8.4 ¿Qué información recibe la IA? (el contexto)

La IA no adivina: recibe lo que **tú ya escribiste** en la Biblia (`SceneContextBuilder`).

| Dato | De dónde sale |
|---|---|
| Esencia del proyecto | Título, género, idioma, premisa, conflicto, tema, tono y ambientación |
| Lugar de la escena | El lugar vinculado a la escena |
| Reglas del mundo | Las del lugar y las de cada personaje, con su rigidez |
| Personajes | Los vinculados a la escena: personalidad, motivación, miedos, límites |
| **Estado previo** de cada personaje | Su emoción en la escena anterior analizada → si no hay, **el estado inicial que definió el autor** → si tampoco, ninguno |
| Escena anterior | Su análisis, para detectar saltos de tono |
| Texto de la escena | La versión **guardada**, conservando quién habla |

Para no gastar de más, el texto se recorta a **1.800 palabras** y se envían como máximo **8 elementos de contexto**. Un análisis típico usa unos 2.500 tokens y tarda unos 2–3 segundos.

### 8.5 ¿Qué responde la IA?

Una respuesta **en formato JSON estricto**, con cuatro partes:

| Parte | Contenido |
|---|---|
| `scene` | Emoción de la escena, resumen, **cita literal** del texto como evidencia y si el cambio de tono respecto a la escena anterior está justificado |
| `characters` | Por cada personaje: emoción, evidencia, si actúa **fuera de personaje** (y por qué) y si su cambio está justificado |
| `inconsistencies` | Contradicciones reales con el lugar, las reglas o la esencia (con gravedad baja, media o alta) |
| `suggestions` | De 1 a 3 sugerencias descartables (emoción, coherencia, ritmo o conflicto) |

**Aunque pidamos formato estricto, siempre se valida en PHP** (`SceneResultValidator`): los números se acotan entre 0 y 1, una emoción inventada se cambia por la más cercana, se descartan personajes que no están en la escena y se recortan los textos largos.

### 8.6 Cómo "mide" emociones sin que el autor vea números (VAD invisible)

Por dentro, cada emoción se describe con **tres números entre 0 y 1**:

| Número | Significa | La interfaz lo llama |
|---|---|---|
| **v** (valencia) | De desagradable a agradable | **Tono** |
| **a** (activación) | De calma total a máxima tensión | **Tensión** |
| **d** (dominancia) | De estar a merced de todo a dominar la situación | **Control del personaje** |

Es el modelo **VAD/PAD** de la psicología (Mehrabian y Russell). Sirve porque permite **medir distancias** entre emociones y comparar de forma objetiva.

El diseñador narrativo no tiene por qué conocer esto, así que la pantalla **nunca muestra** "valencia", "activación" ni "VAD". En su lugar muestra etiquetas:

| Etiqueta | Valores posibles |
|---|---|
| **Tono** | Muy negativo · Negativo · Neutral · Positivo · Muy positivo |
| **Tensión** | Baja · Media · Alta |
| **Control del personaje** | Vulnerable · En equilibrio · Con el control |

Un interruptor de **modo técnico** (apagado por defecto) deja ver los números crudos, útil para una defensa o una depuración.

### 8.7 Las 12 emociones

La lista es **cerrada**. Cada emoción es un punto en el espacio (v, a, d); la emoción de un personaje es **la del punto más cercano** al valor que estimó la IA.

| Emoción | Idea |
|---|---|
| Alegría, Esperanza, Ternura, Calma | Tono positivo (de más a menos activación) |
| Sorpresa, Neutral | Intermedias |
| Ansiedad, Miedo, Tristeza, Culpa | Tono negativo, poco control |
| Ira, Rechazo | Tono negativo, con más control (sobre todo la ira) |

Los valores exactos viven en un único sitio, `config/gamerdust.php`, y el frontend los recibe del servidor. Así hay **una sola fuente de verdad**.

### 8.8 La regla decide, la IA explica

Detectar un **cambio brusco** no se deja a la IA (podría variar de un día a otro). Lo decide una **regla fija**:

```text
cambio = el mayor de |Δv|, |Δa|, |Δd|   (frente al estado previo)
brusco = cambio > 0,40
```

Después, la IA solo aporta lo que una regla no puede: **leer el texto y decir si hay una causa que lo justifique**. El resultado es un semáforo:

| ¿Cambio brusco? (regla) | ¿Justificado? (IA) | Qué ves |
|---|---|---|
| No | — | Nada |
| Sí | **No** | 🔴 **Alerta roja**: cambio brusco sin causa visible |
| Sí | **Sí** | 🟠 **Aviso ámbar**: cambio fuerte, pero el texto lo explica |

Se aplica a cada personaje y también a la escena completa frente a la escena anterior.

### 8.9 ¿Y si la IA falla? (robustez)

| Situación | Qué pasa |
|---|---|
| Todo bien | Se usa el modelo principal (`gpt-oss-120b`) |
| Error, saturación o respuesta inválida | Un reintento con el modelo de respaldo (`gpt-oss-20b`) |
| Falla también el respaldo | Se muestra un **análisis básico por palabras clave** con un aviso claro (`degraded`) |
| Falla al *generar* o *refinar* | No se degrada: mensaje amable y puedes seguir editando |
| No hay clave de Groq | Todo funciona con palabras clave; nunca se bloquea la edición |

**Caché:** si analizas dos veces la misma escena sin cambiar nada (mismo texto, fichas y esencia), se devuelve el resultado guardado **sin llamar a la IA**. *Reanalizar* fuerza una llamada nueva. Si cambias algo, el análisis anterior se marca como **desactualizado** (en gris en los gráficos).

### 8.10 Dónde se ve el resultado

- **Guion → panel de emociones** (`EmotionPanel`): emoción de la escena y de cada personaje con su evidencia, alertas rojas y ámbar, personajes fuera de personaje, incoherencias y sugerencias con botones 👍/👎.
- **Análisis IA → Evolución emocional** (gráficos de Recharts, solo lectura, sin llamar a la IA):
  - **Vista A, arco de la historia:** tono y tensión de cada escena a lo largo del guion.
  - **Vista B, personajes:** cómo cambia la emoción de cada personaje, desde su estado inicial ("Definido por ti") hasta lo detectado en cada escena.
- **Ficha de personaje:** el autor elige una **emoción inicial** y una **intensidad** (baja, media o alta). Sin deslizadores numéricos.

### 8.11 Generar y refinar

- **Borrador con IA** (ficha de personaje): escribes una idea breve y la IA propone una ficha. **Rellena el formulario pero no guarda**, y avisa si el nombre ya existe.
- **Refinar con IA** (un bloque del guion): eliges un objetivo (más tensión, más esperanza, más claro, más conciso o fiel a la personalidad de un personaje). La propuesta aparece aparte; **Aplicar** solo cambia tu borrador local (y se puede deshacer). Nada se guarda hasta que pulses Guardar.

### 8.12 Cuidar el plan gratuito de Groq

| Límite de Groq | Medida de Gamerdust |
|---|---|
| 30 peticiones por minuto | Máximo 10 por usuario y minuto |
| 1.000 peticiones al día | Caché por contenido y llamadas solo bajo demanda |
| 8.000 tokens por minuto | Texto recortado y contexto limitado (~2.500 tokens por análisis) |
| 200.000 tokens al día | Unos 80 análisis diarios por modelo; el respaldo usa la cuota del otro |

### 8.13 ¿Cómo sabemos si la IA ayuda? (evaluación)

- **`ai_generations`** registra cada llamada: tipo, modelo, estado (`ok`, `fallback`, `degraded`, `error`), tokens y latencia. Nunca guarda la clave.
- **`ai_feedback`** guarda los votos 👍/👎 del autor sobre cada propuesta.
- El comando `php artisan gamerdust:ai-metrics --output=metricas.csv` exporta todo a CSV (llamadas, porcentaje de fallos, propuestas útiles, alertas rojas y ámbar…).
- `php artisan gamerdust:groq-check` hace una llamada mínima real para comprobar que la clave, los tokens y la latencia funcionan.

### 8.14 Dónde está en el código

```
app/Contracts/EmotionAnalyzer.php          contrato: "dame el análisis de una escena"
app/Services/Emotion/
├── GroqEmotionAnalyzer.php                implementación con IA real
├── KeywordEmotionAnalyzer.php             implementación por palabras clave (sin red)
├── SceneEmotionService.php                orquesta: caché → analizador → regla → guardar
├── SceneContextBuilder.php                arma el contexto desde la Biblia
├── ScenePrompt.php                        instrucciones y esquema que se envían a la IA
├── SceneResultValidator.php               valida la respuesta
├── EmotionTranslator.php                  números → Tono / Tensión / Control / emoción
├── EmotionChangeDetector.php              la regla del cambio brusco
└── EmotionTimelineService.php             datos para los gráficos
app/Services/Ai/GroqClient.php             llamada HTTP a Groq, reintento y registro
app/Services/Assist/                       generar borrador y refinar bloque
```

**Una decisión de diseño importante:** `EmotionAnalyzer` es un *contrato* con dos implementaciones intercambiables (Groq o palabras clave). Gracias a eso hay degradación, los tests no necesitan internet y, en el futuro, se podría sustituir por un motor Python externo (diseño previsto en `docs/narrative-engine.md`) sin tocar controladores ni interfaz.

### 8.15 Mini demo: el traductor y la regla, en Python

Esta celda **no usa la app**: reproduce la idea de las secciones 8.6 a 8.8 con los mismos números de `config/gamerdust.php`. Puedes cambiar los valores de `antes` y `despues` y ver qué emoción sale y si saltaría una alerta.

In [ ]:
# Demo educativa (no depende del proyecto): emoción más cercana + regla de cambio brusco.
from math import dist

# Centroides (v, a, d) de las 12 emociones, igual que en config/gamerdust.php
EMOCIONES = {
    'alegría':   (0.85, 0.65, 0.65), 'esperanza': (0.75, 0.50, 0.55),
    'ternura':   (0.88, 0.40, 0.55), 'calma':     (0.72, 0.20, 0.60),
    'sorpresa':  (0.60, 0.85, 0.45), 'neutral':   (0.50, 0.40, 0.50),
    'ansiedad':  (0.25, 0.75, 0.30), 'miedo':     (0.10, 0.85, 0.15),
    'tristeza':  (0.15, 0.25, 0.25), 'culpa':     (0.15, 0.45, 0.20),
    'ira':       (0.10, 0.85, 0.75), 'rechazo':   (0.10, 0.55, 0.60),
}
UMBRAL_BRUSCO = 0.40

def emocion_mas_cercana(vad):
    """La emoción cuyo centroide está a menor distancia (euclídea) del valor dado."""
    return min(EMOCIONES, key=lambda nombre: dist(vad, EMOCIONES[nombre]))

def etiqueta(valor, limites, nombres):
    """Convierte un número en una etiqueta usando límites superiores exclusivos."""
    for limite, nombre in zip(limites, nombres):
        if valor < limite:
            return nombre
    return nombres[-1]

def traducir(vad):
    v, a, d = vad
    return {
        'Tono': etiqueta(v, [0.2, 0.4, 0.6, 0.8], ['Muy negativo', 'Negativo', 'Neutral', 'Positivo', 'Muy positivo']),
        'Tensión': etiqueta(a, [0.35, 0.65], ['Baja', 'Media', 'Alta']),
        'Control del personaje': etiqueta(d, [0.35, 0.65], ['Vulnerable', 'En equilibrio', 'Con el control']),
        'Emoción': emocion_mas_cercana(vad),
    }

def cambio_brusco(antes, despues, justificado):
    """La REGLA decide si es brusco; la IA (justificado) solo explica."""
    delta = max(abs(x - y) for x, y in zip(antes, despues))
    if delta <= UMBRAL_BRUSCO:
        return delta, 'sin aviso'
    return delta, 'AMBAR (brusco pero justificado)' if justificado else 'ROJA (brusco y sin causa visible)'

# Ejemplo: un personaje tranquilo que de repente siente miedo.
antes, despues = (0.70, 0.30, 0.60), (0.15, 0.80, 0.20)

print('Antes  :', traducir(antes))
print('Después:', traducir(despues))
for justificado in (False, True):
    delta, alerta = cambio_brusco(antes, despues, justificado)
    print(f'Justificado por la IA = {justificado!s:5} -> cambio {delta:.2f} -> alerta {alerta}')

## 9. Lo que todavía usa reglas simples (sin IA)

No todo es IA. Estas piezas usan reglas fijas, y se muestran en el Panel y en Análisis IA:

| Pieza | Qué hace |
|---|---|
| **Validación de coherencia** (`NarrativeValidationService`) | Avisa de: proyecto sin premisa, personaje sin motivación, escena sin conflicto, lore sin función narrativa, regla *inquebrantable* que un personaje rompe, y regla *rompible a un coste* sin coste definido. Al recalcular, borra y recrea los avisos no resueltos. |
| **Estimación de coste** (`NarrativeCostEstimatorService`) | Puntúa escenas y diálogos (bajo, medio, alto) según decisiones del jugador, consecuencias, número de diálogos y ramificación. Se calcula al crear o editar. |
| **Análisis de nodos de diálogo** (`EmotionalAnalysisService`) | Cuenta palabras clave. Pendiente de pasar a Groq. |

El análisis de **escenas** del Guion sí usa la IA (sección 8); si Groq falla o no hay clave, cae en palabras clave y lo indica.

## 10. Cómo instalar, ejecutar y probar

**Requisitos:** PHP 8.4+ (con la extensión `zip`), Composer, Node 22+ y MySQL. En Laragon, elige PHP 8.4 o superior y apunta el sitio a `public/`.

Desde la raíz del repositorio:

```bash
composer setup            # instala todo, crea .env, clave, migra y compila
# o, paso a paso:
composer install
npm install
php artisan key:generate
php artisan migrate --seed
```

**Para trabajar** (dos terminales, o `composer dev` que lo hace todo junto):

```bash
php artisan serve         # http://localhost:8000
npm run dev               # Vite, solo sirve los archivos del frontend
```

Abre la URL de Laravel (`localhost:8000`), **no** el puerto 5173 de Vite.

**Activar la IA real** (opcional): añade en `.env`

```
GROQ_API_KEY=tu_clave
```

Sin clave, la app funciona con palabras clave. Otras variables: `GROQ_MODEL`, `GROQ_FALLBACK_MODEL`, `GROQ_TIMEOUT`, `GAMERDUST_AI_DRIVER`, `GAMERDUST_AI_MAX_TOKENS`.

**Tests y utilidades**

```bash
php artisan test                  # 112 tests (usan SQLite en memoria, nunca Groq real)
php artisan gamerdust:groq-check  # comprueba la conexión real con Groq
vendor/bin/pint                   # formatea el código PHP
npx tsc --noEmit                  # revisa los tipos de TypeScript
npm run build                     # compila el frontend para producción
```

**Problemas frecuentes**

- **Pantalla en blanco tras un fallo de `npm run dev`:** borra `public/hot` y reinicia Vite.
- **Vite se queda sin memoria** (sobre todo tras `npm install`): cierra programas, o usa `npm run build` y no `npm run dev` si no estás editando el frontend.
- **Fallan 5 tests con `APP_LOCALE=en`:** esperan textos en español. Ejecuta `APP_LOCALE=es php artisan test`.
- **`gamerdustv1.test` muestra un listado de carpetas:** la raíz del sitio debe ser `public/`.

## 11. Estado actual y qué falta

**Funciona hoy**

- Cuentas, proyectos (género, idioma, favoritos) y restricciones creativas.
- Biblia Narrativa completa: esencia, personajes (con imagen y estado emocional inicial), lugares y reglas con sus vínculos.
- Editor de guion con versiones, actos, impresión y control de conflictos.
- Editor de diálogos ramificados con `{{referencias}}`.
- Mecánicas, comentarios y reportes en Markdown.
- **IA emocional con Groq**: análisis de escena, alertas rojas y ámbar, gráficos de evolución, borrador de personaje, refinado de bloques, valoraciones 👍/👎 y métricas.
- Interfaz en español e inglés.
- 112 tests automáticos.

**Pendiente**

- Motor Python: no existe; su diseño previsto está en `docs/narrative-engine.md`.
- Análisis con Groq de los nodos del editor de diálogos.
- Validación de coherencia más profunda (continuidad entre escenas, arcos).
- Pruebas de recorrido completo (`tests/e2e/`, vacío).
- Chat conversacional, extracción automática de entidades, análisis de audio e imagen (fuera de alcance por ahora).
- Verificación real del correo al registrarse (hoy los usuarios se marcan como verificados al crearse).

## 12. Glosario rápido

| Término | Significado |
|---|---|
| **Biblia Narrativa** | Conjunto de fichas (esencia, personajes, lugares, reglas) que describen el mundo de la historia |
| **Lore** | Información del mundo: lugares, reglas, facciones, objetos, eventos |
| **Escena** | Unidad de la historia con su guion, lugar y personajes |
| **Nodo / pasaje** | Caja de diálogo en el editor de ramificación |
| **Inertia** | Pegamento que permite usar React con Laravel sin crear una API aparte |
| **LLM** | Modelo de lenguaje grande: la IA que lee y escribe texto |
| **Groq** | Servicio externo que ejecuta los modelos de lenguaje |
| **gpt-oss** | Familia de modelos de lenguaje que se usa a través de Groq |
| **VAD** | Tres dimensiones emocionales: valencia, activación, dominancia |
| **Centroide** | Punto "típico" de una emoción en el espacio v, a, d |
| **Contexto** | Lo que se le da a la IA para que no adivine: fichas, lugar, reglas |
| **Token** | Trozo de texto que cuenta para el límite y el coste de la IA |
| **Caché** | Resultado guardado para no repetir una llamada igual |
| **Degradación** | Plan B cuando la IA falla: análisis básico por palabras clave |
| **Hash** | "Huella" de un contenido: si cambia el texto, cambia el hash |
| **Alerta roja / ámbar** | Cambio emocional brusco sin causa visible / brusco pero justificado |

In [ ]:
# Celda opcional: inventario rápido de los archivos clave de la app (raíz del repositorio).
from pathlib import Path

def find_api_root(start=None):
    """Sube desde el directorio actual hasta encontrar el proyecto Laravel (el que tiene artisan)."""
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'artisan').exists():
            return candidate
    raise FileNotFoundError('No se encontró artisan. Ejecuta el cuaderno dentro del repositorio.')

root = find_api_root()
carpetas = [
    'app/Http/Controllers',
    'app/Models',
    'app/Services/Emotion',
    'app/Services/Ai',
    'app/Services/Assist',
    'resources/js/features',
    'database/migrations',
    'tests',
]

print('Raíz de la API:', root)
for carpeta in carpetas:
    ruta = root / carpeta
    archivos = [p for p in ruta.rglob('*') if p.is_file()] if ruta.is_dir() else []
    print(f'{carpeta:<32} {len(archivos):>3} archivos')